In [2]:
import nba_api.stats.endpoints as endpoints
import nba_api.stats.static.players as players
import nba_api.stats.static.teams as teams
import pandas as pd
import os
import time
from requests.exceptions import HTTPError
from pprint import pprint
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import (
    ColumnTransformer,
    TransformedTargetRegressor,
    make_column_transformer,
)
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, RidgeClassifier
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [3]:
players_list = players.get_players()
team_list = teams.get_teams()
active_players_list = players.get_active_players()
playercareerstats = endpoints.playercareerstats
playerawards = endpoints.playerawards
playerestimatedmetrics = endpoints.playerestimatedmetrics
leaguedashplayerstats = endpoints.leaguedashplayerstats
teamplayeronoffdetails = endpoints.teamplayeronoffdetails

In [4]:
# --------------------------------------------------------
# Helper: cached NBA API fetch with retry and backoff
# --------------------------------------------------------
def get_cached_data(endpoint_func, season, per_mode="Per100Possessions", cache_dir="cache", label="stats"):
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, f"{label}_{season}.csv")

    if os.path.exists(cache_path):
        return pd.read_csv(cache_path)

    retries = 5
    for attempt in range(retries):
        try:
            if label == "stats":
                df = endpoint_func(season=season, per_mode_detailed=per_mode).get_data_frames()[0]
            if label == "on_off":
                df = get_season_on_off(season=season, per_mode=per_mode)
            else:
                df = endpoint_func(season=season).get_data_frames()[0]

            df.to_csv(cache_path, index=False)
            time.sleep(1.0)
            return df

        except HTTPError as e:
            if e.response.status_code == 429:
                wait = 10 * (attempt + 1)
                print(f"Rate limited on {label} {season}, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise e

    raise RuntimeError(f"Failed to fetch {label} for {season} after {retries} retries.")

# Helper to get on off stats with caching
def get_season_on_off(season, per_mode, cache_dir="cache"):
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, f"on_off_{season}.csv")

    if os.path.exists(cache_path):
        return pd.read_csv(cache_path)

    # print("starting season on off")
    season_on_off = pd.DataFrame()
    columns_to_remove = [
        "GROUP_SET", "TEAM_NAME", "COURT_STATUS", "VS_PLAYER_NAME", "TEAM_ID", "TEAM_ABBREVIATION", "COURT_STATUS", "W", "L", "W_PCT",
        'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK',
        'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK',
        'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK'
    ]
    for team in team_list:
        # print("starting api call")
        team_on_off_details = teamplayeronoffdetails.TeamPlayerOnOffDetails(team_id=team["id"], per_mode_detailed=per_mode, season=season)
        # print("finished api call")
        team_on = team_on_off_details.get_data_frames()[1]
        team_on.drop(columns=columns_to_remove, inplace=True)
        team_off = team_on_off_details.get_data_frames()[2]
        team_off.drop(columns=columns_to_remove, inplace=True)
        team_on_off = team_on.merge(team_off, on="VS_PLAYER_ID", suffixes=("_ON", "_OFF"))
        team_on_off.drop(columns=["GP_OFF"], inplace=True)
        team_on_off = team_on_off.rename(columns={"VS_PLAYER_ID": "PLAYER_ID", "GP_ON": "GP"})
        season_on_off = weighted_df_merge(season_on_off, team_on_off)
    # print("finished season on off")
    season_on_off.to_csv(cache_path, index=False)
    return season_on_off

def weighted_df_merge(df1, df2):
    # Concatenate the two dataframes
    # print("started df merge")
    df = pd.concat([df1, df2], ignore_index=True)
    if df.empty:
        return df

    # Identify numeric columns (excluding PLAYER_ID and VS_PLAYER_ID)
    exclude_cols = {"PLAYER_ID", "VS_PLAYER_ID"}
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) and col not in exclude_cols]

    # Group by PLAYER_ID and apply weighted average for numeric columns
    def weighted_avg(group):
        total_games = group["GP"].sum() if "GP" in group else 1
        result = {}
        for col in group.columns:
            if col in exclude_cols:
                result[col] = group.iloc[0][col]
            elif col == "GP":
                result[col] = total_games
            elif col in numeric_cols:
                # Weighted average by GP, fallback to mean if GP is zero
                if total_games > 0:
                    result[col] = (group[col] * group["GP"]).sum() / total_games
                else:
                    result[col] = group[col].mean()
            else:
                result[col] = group.iloc[0][col]
        return pd.Series(result)

    merged_df = df.groupby("PLAYER_ID", as_index=False).apply(weighted_avg)
    # print("finished df merge")
    return merged_df.reset_index(drop=True)

def merge_player_rows(rows):
    # This function is not needed with the new weighted_df_merge logic,
    # but kept for compatibility if called elsewhere.
    return rows

# --------------------------------------------------------
# Helper: merge stats + estimated metrics for one season
# --------------------------------------------------------
def get_season_data(season, suffix=""):
    stats = get_cached_data(leaguedashplayerstats.LeagueDashPlayerStats, season, label="stats")
    est = get_cached_data(playerestimatedmetrics.PlayerEstimatedMetrics, season, label="est")
    on_off = get_cached_data(teamplayeronoffdetails.TeamPlayerOnOffDetails, season, label="on_off")
    totals = get_cached_data(leaguedashplayerstats.LeagueDashPlayerStats, season, per_mode="Totals", label="totals")
    total_min = totals[["PLAYER_ID", "MIN"]]

    df = stats.merge(est, on="PLAYER_ID", suffixes=("", "_EST"))
    df = df.merge(on_off, on="PLAYER_ID", suffixes=("", "_ON_OFF"))
    df = df.merge(total_min, on="PLAYER_ID", suffixes=("", "_TOTAL"))
    df["Season"] = season

    # clean column naming for standardization
    df.columns = [c.strip().replace(" ", "_") for c in df.columns]
    df = df.add_suffix(suffix)
    df.rename(columns={f"PLAYER_ID{suffix}": "PLAYER_ID", f"Season{suffix}": "Season"}, inplace=True)
    return df


In [37]:
fourfactors.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS',
       'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK',
       'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK',
       'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK',
       'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK',
       'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK',
       'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK',
       'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT'],
      dtype='object')

In [5]:
reg_season_2026 = get_season_data("2025-26")

In [6]:
reg_season_2022 = get_season_data("2021-22")
print("gotten on/off for 22")
reg_season_2023 = get_season_data("2022-23")
print("gotten on/off for 23")
reg_season_2024 = get_season_data("2023-24")
print("gotten on/off for 24")
reg_season_2025 = get_season_data("2024-25")
print("gotten on/off for 25")
reg_season_2026 = get_season_data("2025-26")
print("gotten on/off for 26")

gotten on/off for 22
gotten on/off for 23
gotten on/off for 24
gotten on/off for 25
gotten on/off for 26


In [7]:


training_df = pd.concat([reg_season_2022, reg_season_2023, reg_season_2024, reg_season_2025], axis=0, ignore_index=True)



In [8]:
# remove players with less than 500 minutes played
filtered_training_df = training_df[training_df['MIN_TOTAL'] > 500]
filtered_test_df = reg_season_2026[reg_season_2026['MIN_TOTAL'] > 500]
print(reg_season_2026.columns.to_list())

['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT', 'PLAYER_NAME_EST', 'GP_EST', 'W_EST', 'L_EST', 'W_PCT_EST', 'MIN_EST', 'E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_AST_RATIO', 'E_OREB_PCT', 'E_DREB_PCT', 'E_REB_PCT', 'E_TOV_PCT', 'E_USG_PCT', 'E_PACE', 'GP_RANK_EST', 'W_RANK_EST', 'L_RANK_ES

In [9]:
filtered_training_df["ON_OFF"] = filtered_training_df["PLUS_MINUS_ON"] - filtered_training_df["PLUS_MINUS_OFF"]
filtered_test_df["ON_OFF"] = filtered_test_df["PLUS_MINUS_ON"] - filtered_test_df["PLUS_MINUS_OFF"]

/var/folders/p0/2rx3rhtn7xdfkgbkd_060nsw0000gn/T/ipykernel_99868/2972717926.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_training_df["ON_OFF"] = filtered_training_df["PLUS_MINUS_ON"] - filtered_training_df["PLUS_MINUS_OFF"]
/var/folders/p0/2rx3rhtn7xdfkgbkd_060nsw0000gn/T/ipykernel_99868/2972717926.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_test_df["ON_OFF"] = filtered_test_df["PLUS_MINUS_ON"] - filtered_test_df["PLUS_MINUS_OFF"]


In [10]:
target_name = "ON_OFF"
columns_to_remove = "E_NET_RATING_RANK,GP_RANK_EST,W_RANK_EST,L_RANK_EST,W_PCT_RANK_EST,MIN_RANK_EST,E_OFF_RATING_RANK,E_DEF_RATING_RANK,E_AST_RATIO_RANK,E_OREB_PCT_RANK,E_DREB_PCT_RANK,E_REB_PCT_RANK,E_TOV_PCT_RANK,E_USG_PCT_RANK,E_PACE_RANK,Season, NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT,PLAYER_NAME_EST, GP_ON_OFF, MIN_ON, PLUS_MINUS, MIN_OFF,MIN_TOTAL"
columns_to_remove = columns_to_remove.replace(" ", "").split(",")

# diff = list(set(filtered_test_df.columns.to_list()) - set(columns_to_remove))
cols = filtered_test_df.columns.to_list()
print(cols)
for i in columns_to_remove:
    if i not in cols:
        print(i)


filtered_training_df.drop(columns=columns_to_remove, inplace=True)
filtered_test_df.drop(columns=columns_to_remove, inplace=True)
print(filtered_test_df.columns.to_list())

['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT', 'PLAYER_NAME_EST', 'GP_EST', 'W_EST', 'L_EST', 'W_PCT_EST', 'MIN_EST', 'E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_AST_RATIO', 'E_OREB_PCT', 'E_DREB_PCT', 'E_REB_PCT', 'E_TOV_PCT', 'E_USG_PCT', 'E_PACE', 'GP_RANK_EST', 'W_RANK_EST', 'L_RANK_ES

/var/folders/p0/2rx3rhtn7xdfkgbkd_060nsw0000gn/T/ipykernel_99868/2827268138.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_training_df.drop(columns=columns_to_remove, inplace=True)
/var/folders/p0/2rx3rhtn7xdfkgbkd_060nsw0000gn/T/ipykernel_99868/2827268138.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_test_df.drop(columns=columns_to_remove, inplace=True)


In [11]:
# train_df, test_df = train_test_split(reg_season_2026, test_size=0.2, random_state=123)
train_df = filtered_training_df
test_df = filtered_test_df
X_train = train_df.drop(target_name, axis=1)
y_train = train_df[target_name]
X_test = test_df.drop(target_name, axis=1)
y_test = test_df[target_name]

In [12]:
print(train_df.columns.to_list())

['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'GP_EST', 'W_EST', 'L_EST', 'W_PCT_EST', 'MIN_EST', 'E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_AST_RATIO', 'E_OREB_PCT', 'E_DREB_PCT', 'E_REB_PCT', 'E_TOV_PCT', 'E_USG_PCT', 'E_PACE', 'FGM_ON', 'FGA_ON', 'FG_PCT_ON', 'FG3M_ON', 'FG3A_ON', 'FG3_PCT_ON', 'FTM_ON', 'FTA_ON', 'FT_PCT_ON', 'OREB_ON', 'DREB_ON', 'REB_ON', 'AST_ON', 'TOV_ON', 'STL_ON', 'BLK_ON', 'BLKA_ON', 'PF_ON', 'PFD_ON', 'PTS_ON', 'PLUS_MINUS_ON', 'FGM_OFF', 'FGA_OFF', 'FG_PCT_OFF', 'FG3M_OFF', 'FG3A_OFF', 'FG3_PCT_OFF', 'FTM_OFF', 'FTA_OFF', 'FT_PCT_OFF', 'OREB_OFF', 'DREB_OFF', 'REB_OFF', 'AST_OFF', 'TOV_OFF', 'STL_OFF', 'BLK_OFF', 'BLKA_OFF', 'PF_OFF', 'PFD_OFF', 'PTS_OFF', 'PLUS_MINUS_OFF', 'ON_OFF']


In [31]:
for df in (train_df, test_df):
    denom = (df["FGA_ON"] + 0.44 * df["FTA_ON"] + df["TOV_ON"])
    df["TOV_PCT_ON"] = (df["TOV_ON"]) / denom.replace({0: pd.NA})
    denom = (df["FGA_OFF"] + 0.44 * df["FTA_OFF"] + df["TOV_OFF"])
    df["TOV_PCT_OFF"] = (df["TOV_OFF"]) / denom.replace({0: pd.NA})

/var/folders/p0/2rx3rhtn7xdfkgbkd_060nsw0000gn/T/ipykernel_99868/1438287606.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["TOV_PCT_ON"] = (df["TOV_ON"]) / denom.replace({0: pd.NA})
/var/folders/p0/2rx3rhtn7xdfkgbkd_060nsw0000gn/T/ipykernel_99868/1438287606.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["TOV_PCT_OFF"] = (df["TOV_OFF"]) / denom.replace({0: pd.NA})


In [32]:
for df in (train_df, test_df):
    denom = 2 * (df["FGA_ON"] + 0.44 * df["FTA_ON"])
    df["TS_ON"] = df["PTS_ON"] / denom.replace({0: pd.NA})
    denom = 2 * (df["FGA_OFF"] + 0.44 * df["FTA_OFF"])
    df["TS_OFF"] = df["PTS_OFF"] / denom.replace({0: pd.NA})

/var/folders/p0/2rx3rhtn7xdfkgbkd_060nsw0000gn/T/ipykernel_99868/1351446076.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["TS_ON"] = df["PTS_ON"] / denom.replace({0: pd.NA})
/var/folders/p0/2rx3rhtn7xdfkgbkd_060nsw0000gn/T/ipykernel_99868/1351446076.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["TS_OFF"] = df["PTS_OFF"] / denom.replace({0: pd.NA})


In [33]:
df

,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,BLKA_OFF,PF_OFF,PFD_OFF,PTS_OFF,PLUS_MINUS_OFF,ON_OFF,TS_ON,TS_OFF,TOV_PCT_ON,TOV_PCT_OFF
1,1631260,AJ Green,AJ,1610612749,MIL,26.0,78,31,47,0.397,...,3.6,20.1,17.3,107.4,-10.3,7.2,0.594262,0.582126,0.135381,0.144648
3,203932,Aaron Gordon,Aaron,1610612743,DEN,30.0,36,27,9,0.750,...,4.5,19.6,20.9,119.3,2.4,10.6,0.621263,0.613759,0.093078,0.122757
4,1628988,Aaron Holiday,Aaron,1610612745,HOU,29.0,57,38,19,0.667,...,5.0,18.2,19.9,117.3,4.6,3.4,0.572748,0.576401,0.128870,0.134409
5,1630174,Aaron Nesmith,Aaron,1610612754,IND,26.0,45,10,35,0.222,...,4.7,19.6,19.1,108.0,-8.1,1.1,0.571032,0.566346,0.121070,0.131210
6,1630598,Aaron Wiggins,Aaron,1610612760,OKC,27.0,65,50,15,0.769,...,4.1,18.9,20.1,118.8,13.7,-7.7,0.584989,0.607238,0.121226,0.107645
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
574,1642258,Zaccharie Risacher,Zaccharie,1610612737,ATL,21.0,67,39,28,0.582,...,4.7,18.9,17.6,114.8,4.1,-4.6,0.577112,0.588259,0.123736,0.121538
577,203897,Zach LaVine,Zach,1610612758,SAC,31.0,39,9,30,0.231,...,4.7,19.2,18.2,110.3,-8.9,-3.2,0.564147,0.558719,0.127059,0.125766
578,1630192,Zeke Nnaji,Zeke,1610612743,DEN,25.0,52,38,14,0.731,...,4.3,18.8,21.8,123.2,7.6,-16.1,0.569414,0.624215,0.139138,0.110827
579,1630533,Ziaire Williams,Ziaire,1610612751,BKN,24.0,56,13,43,0.232,...,5.3,21.3,19.6,109.5,-9.9,-0.6,0.550442,0.564131,0.159710,0.133866


In [13]:
dummy = DummyRegressor()
dummy_scores = cross_validate(dummy, X_train, y_train, return_train_score=True)
pd.DataFrame(dummy_scores)

,fit_time,score_time,test_score,train_score
0,0.001755,0.000281,-0.005249,0.0
1,0.000691,0.000225,-0.000085,0.0
2,0.001346,0.000619,-0.010789,0.0
3,0.000486,0.000177,-0.000063,0.0
4,0.000378,0.000156,-0.002853,0.0


In [14]:
reg_season_2026.to_csv("reg_season_2026_stats.csv", index=False)

In [15]:
numeric_transformer = make_pipeline(StandardScaler())
# numeric_feats = "FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,E_OREB_PCT,E_DREB_PCT,E_REB_PCT,E_TOV_PCT,E_USG_PCT,FGM_ON,FGA_ON,FG_PCT_ON,FG3M_ON,FG3A_ON,FG3_PCT_ON,FTM_ON,FTA_ON,FT_PCT_ON,OREB_ON,DREB_ON,REB_ON,AST_ON,TOV_ON,STL_ON,BLK_ON,BLKA_ON,PF_ON,PFD_ON,PTS_ON,FGM_OFF,FGA_OFF,FG_PCT_OFF,FG3M_OFF,FG3A_OFF,FG3_PCT_OFF,FTM_OFF,FTA_OFF,FT_PCT_OFF,OREB_OFF,DREB_OFF,REB_OFF,AST_OFF,TOV_OFF,STL_OFF,BLK_OFF,BLKA_OFF,PF_OFF,PFD_OFF,PTS_OFF".split(",")
numeric_feats = "FGM_ON,FGA_ON,FG_PCT_ON,FG3M_ON,FG3A_ON,FG3_PCT_ON,FTM_ON,FTA_ON,FT_PCT_ON,OREB_ON,DREB_ON,REB_ON,AST_ON,TOV_ON,STL_ON,BLK_ON,BLKA_ON,PF_ON,PFD_ON,PTS_ON,FGM_OFF,FGA_OFF,FG_PCT_OFF,FG3M_OFF,FG3A_OFF,FG3_PCT_OFF,FTM_OFF,FTA_OFF,FT_PCT_OFF,OREB_OFF,DREB_OFF,REB_OFF,AST_OFF,TOV_OFF,STL_OFF,BLK_OFF,BLKA_OFF,PF_OFF,PFD_OFF,PTS_OFF".split(",")
drop_feats = list(dict.fromkeys(
    "PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,GP_EST,PLUS_MINUS_ON,PLUS_MINUS_OFF,E_OFF_RATING,E_DEF_RATING,MIN,MIN_EST,W_PCT_EST,L_EST,W_EST,W,L,W_PCT,E_PACE,E_AST_RATIO".split(",")
))

preprocessor = make_column_transformer(
    (numeric_transformer, numeric_feats),   
    ("drop", drop_feats), 
    verbose_feature_names_out = False
)

regression_pipe =  make_pipeline(preprocessor, SVR(C=10.0, gamma=0.008, epsilon=0.001))
# regression_pipe =  make_pipeline(preprocessor, Ridge(alpha=0.01, max_iter=1000, random_state=123))
# regression_pipe =  make_pipeline(preprocessor, RandomForestRegressor(max_features=10, n_estimators=100, max_depth=3, random_state=123))


In [26]:
cols = X_train.columns.to_list()
print(cols)


['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'GP_EST', 'W_EST', 'L_EST', 'W_PCT_EST', 'MIN_EST', 'E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_AST_RATIO', 'E_OREB_PCT', 'E_DREB_PCT', 'E_REB_PCT', 'E_TOV_PCT', 'E_USG_PCT', 'E_PACE', 'FGM_ON', 'FGA_ON', 'FG_PCT_ON', 'FG3M_ON', 'FG3A_ON', 'FG3_PCT_ON', 'FTM_ON', 'FTA_ON', 'FT_PCT_ON', 'OREB_ON', 'DREB_ON', 'REB_ON', 'AST_ON', 'TOV_ON', 'STL_ON', 'BLK_ON', 'BLKA_ON', 'PF_ON', 'PFD_ON', 'PTS_ON', 'PLUS_MINUS_ON', 'FGM_OFF', 'FGA_OFF', 'FG_PCT_OFF', 'FG3M_OFF', 'FG3A_OFF', 'FG3_PCT_OFF', 'FTM_OFF', 'FTA_OFF', 'FT_PCT_OFF', 'OREB_OFF', 'DREB_OFF', 'REB_OFF', 'AST_OFF', 'TOV_OFF', 'STL_OFF', 'BLK_OFF', 'BLKA_OFF', 'PF_OFF', 'PFD_OFF', 'PTS_OFF', 'PLUS_MINUS_OFF']


In [27]:
all_feats = set(numeric_feats + drop_feats)
# missing = all_feats - set(X_train.columns)
print(all_feats)

{'MIN_EST', 'GP', 'FT_PCT', 'PFD', 'FG3M_OFF', 'E_OFF_RATING', 'FG_PCT', 'PLUS_MINUS_OFF', 'E_TOV_PCT', 'TOV_ON', 'FTA', 'FT_PCT_OFF', 'FGA_ON', 'PTS', 'E_REB_PCT', 'PF', 'FGM_OFF', 'GP_EST', 'PFD_ON', 'BLKA_OFF', 'BLK_OFF', 'FGM_ON', 'L_EST', 'FG3_PCT_OFF', 'NICKNAME', 'PLAYER_NAME', 'FTA_ON', 'E_AST_RATIO', 'FG3_PCT', 'MIN', 'E_PACE', 'REB_OFF', 'FGA', 'PLUS_MINUS_ON', 'STL_ON', 'DREB_ON', 'TOV', 'E_OREB_PCT', 'W_PCT_EST', 'FG3A', 'AST_ON', 'E_DREB_PCT', 'FGM', 'FTA_OFF', 'OREB', 'FG_PCT_OFF', 'BLK_ON', 'BLK', 'BLKA', 'FG3M', 'REB_ON', 'FG3_PCT_ON', 'FTM_OFF', 'BLKA_ON', 'PTS_ON', 'L', 'FG_PCT_ON', 'FTM', 'PFD_OFF', 'W_PCT', 'PLAYER_ID', 'FG3A_ON', 'TEAM_ABBREVIATION', 'FT_PCT_ON', 'OREB_ON', 'PTS_OFF', 'W', 'FTM_ON', 'FG3M_ON', 'OREB_OFF', 'PF_ON', 'TOV_OFF', 'E_DEF_RATING', 'FGA_OFF', 'AST', 'E_USG_PCT', 'AST_OFF', 'DREB_OFF', 'DREB', 'STL_OFF', 'REB', 'PF_OFF', 'W_EST', 'TEAM_ID', 'AGE', 'STL', 'FG3A_OFF'}


In [18]:
reg_scores = cross_validate(regression_pipe, X_train, y_train, return_train_score=True)
reg_scores

{'fit_time': array([0.06929898, 0.06335473, 0.06549025, 0.06233406, 0.0605998 ]),
 'score_time': array([0.028543  , 0.02894521, 0.02646875, 0.02324295, 0.02641416]),
 'test_score': array([0.69990522, 0.72333701, 0.74710157, 0.75619454, 0.68355738]),
 'train_score': array([0.87213021, 0.87202315, 0.87164414, 0.86390951, 0.87845227])}

In [19]:
regression_pipe.fit(X_train, y_train)

,steps,"[('columntransformer', ...), ('svr', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline', ...), ('drop', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [20]:
# X = pd.concat([X_test, X_train])

In [21]:
predictions = regression_pipe.predict(X_test)
results_df = X_test.assign(Predicted_Value=predictions)

In [22]:
results_df = results_df[['PLAYER_NAME', 'Predicted_Value']]
df_sorted = results_df.sort_values(by='Predicted_Value', ascending=False)
df_sorted

,PLAYER_NAME,Predicted_Value
237,Jalen Smith,13.269831
186,Giannis Antetokounmpo,12.057283
559,Victor Wembanyama,11.911322
429,Nikola Jokić,11.783922
501,Shai Gilgeous-Alexander,11.083101
...,...,...
520,Terrence Shannon Jr.,-9.578861
459,Patrick Williams,-10.314049
328,Kenrich Williams,-10.320033
535,Tre Mann,-10.843237


In [23]:
df_sorted.to_csv("model_results.csv")

In [24]:
df_sorted["PLAYER_NAME"].to_csv("model_results_name_only.csv", index=False)

In [25]:
regression_pipe_coefs = pd.DataFrame(
    data=regression_pipe.named_steps["ridge"].coef_, index=numeric_feats, columns=["Coefficient"]
)

KeyError: 'ridge'

In [ ]:
# regression_pipe_coefs.sort_values(by="Coefficient")
regression_pipe_coefs.sort_values(by="Coefficient", ascending=False).head(80)

,Coefficient
E_REB_PCT,18.728728
PTS_ON,7.128917
E_USG_PCT,5.290409
FG_PCT_ON,4.536353
FTM_OFF,3.893504
...,...
REB_OFF,-5.646199
PTS_OFF,-6.439293
FGM_ON,-7.596491
E_DREB_PCT,-8.073956
